In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Data Loading

Configure the path to your results directory:

In [ ]:
# Configure your results directory here
RESULTS_DIR = Path("../../RESULTS/NSGAII/DTLZ3")

# Verify path exists
if not RESULTS_DIR.exists():
    print(f"WARNING: Directory not found: {RESULTS_DIR}")
    print("Please update RESULTS_DIR to point to your results.")
else:
    print(f"Results directory: {RESULTS_DIR.resolve()}")
    print(f"Files found: {len(list(RESULTS_DIR.glob('*')))}")

In [ ]:
def load_configurations(results_dir: Path):
    """Load all configuration files and objective values."""
    configs = []
    objectives = []
    
    config_pattern = re.compile(r"VAR\..*\.Conf\.DoubleValues\.(\d+)\.csv")
    fun_pattern = re.compile(r"FUN\..*\.(\d+)\.csv")
    
    for config_file in sorted(results_dir.glob("VAR.*.Conf.DoubleValues.*.csv")):
        match = config_pattern.search(config_file.name)
        if match:
            eval_num = int(match.group(1))
            df = pd.read_csv(config_file)
            df["evaluation"] = eval_num
            configs.append(df)
    
    for fun_file in sorted(results_dir.glob("FUN.*.csv")):
        match = fun_pattern.search(fun_file.name)
        if match:
            eval_num = int(match.group(1))
            obj_df = pd.read_csv(fun_file, header=None, names=["Epsilon", "NormHypervolume"])
            obj_df["evaluation"] = eval_num
            objectives.append(obj_df)
    
    configs_df = pd.concat(configs, ignore_index=True)
    objectives_df = pd.concat(objectives, ignore_index=True)
    
    return configs_df, objectives_df

# Load data
configs_df, objectives_df = load_configurations(RESULTS_DIR)
print(f"Configurations loaded: {len(configs_df)}")
print(f"Evaluations: {configs_df['evaluation'].nunique()}")
print(f"Parameters: {len(configs_df.columns) - 1}")

In [ ]:
# Preview configuration data
configs_df.head()

In [ ]:
# Preview objectives
objectives_df.head()

## 2. Correlation Analysis

Analyze correlations between parameters and objective values.

In [ ]:
# Merge configurations with objectives
obj_means = objectives_df.groupby("evaluation").mean().reset_index()
merged = configs_df.merge(obj_means, on="evaluation", how="inner")

# Calculate correlations
param_cols = [col for col in configs_df.columns if col != "evaluation"]

correlations = {}
for param in param_cols:
    if merged[param].std() > 0:
        corr_eps, p_eps = stats.spearmanr(merged[param], merged["Epsilon"])
        corr_nhv, p_nhv = stats.spearmanr(merged[param], merged["NormHypervolume"])
        correlations[param] = {
            "Epsilon_corr": corr_eps,
            "Epsilon_pvalue": p_eps,
            "NormHV_corr": corr_nhv,
            "NormHV_pvalue": p_nhv,
        }

corr_df = pd.DataFrame(correlations).T
corr_df["abs_mean_corr"] = (corr_df["Epsilon_corr"].abs() + corr_df["NormHV_corr"].abs()) / 2
corr_df = corr_df.sort_values("abs_mean_corr", ascending=False)

print("Top 15 parameters by correlation with objectives:")
corr_df.head(15)[["Epsilon_corr", "NormHV_corr", "abs_mean_corr"]]

In [ ]:
# Correlation heatmap
top_params = corr_df.head(12).index.tolist()

plt.figure(figsize=(12, 10))
corr_matrix = merged[top_params + ["Epsilon", "NormHypervolume"]].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5)
plt.title("Parameter-Objective Correlation Matrix")
plt.tight_layout()
plt.show()

## 3. PCA Analysis

Reduce dimensionality to understand the configuration space structure.

In [ ]:
# Prepare data for PCA
non_constant = [col for col in param_cols if configs_df[col].std() > 0]
X = configs_df[non_constant].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Perform PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Explained variance
cumsum = np.cumsum(pca.explained_variance_ratio_)
n_90 = np.argmax(cumsum >= 0.90) + 1
n_95 = np.argmax(cumsum >= 0.95) + 1

print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")
print(f"Total non-constant parameters: {len(non_constant)}")

In [ ]:
# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1), 
            pca.explained_variance_ratio_, alpha=0.7, color='steelblue')
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("Scree Plot")

axes[1].plot(range(1, len(cumsum) + 1), cumsum, 'b-o', markersize=4)
axes[1].axhline(y=0.90, color='r', linestyle='--', label='90% variance')
axes[1].axhline(y=0.95, color='g', linestyle='--', label='95% variance')
axes[1].axvline(x=n_90, color='r', linestyle=':', alpha=0.5)
axes[1].axvline(x=n_95, color='g', linestyle=':', alpha=0.5)
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].set_title("Cumulative Explained Variance")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# PCA loadings
loadings = pd.DataFrame(
    pca.components_[:5].T,
    columns=[f"PC{i+1}" for i in range(5)],
    index=non_constant
)
loadings["PC1_abs"] = loadings["PC1"].abs()
loadings = loadings.sort_values("PC1_abs", ascending=False)

print("Top loadings for first 3 PCs:")
loadings.head(10)[["PC1", "PC2", "PC3"]]

In [ ]:
# PCA scatter plot colored by objective
merged_params = merged[non_constant].values
merged_scaled = scaler.transform(merged_params)
merged_pca = pca.transform(merged_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc1 = axes[0].scatter(merged_pca[:, 0], merged_pca[:, 1], 
                      c=merged["Epsilon"], cmap="viridis", alpha=0.7, s=50)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[0].set_title("Configuration Space - Colored by Epsilon")
plt.colorbar(sc1, ax=axes[0], label="Epsilon (lower is better)")

sc2 = axes[1].scatter(merged_pca[:, 0], merged_pca[:, 1], 
                      c=merged["NormHypervolume"], cmap="viridis", alpha=0.7, s=50)
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[1].set_title("Configuration Space - Colored by NormHypervolume")
plt.colorbar(sc2, ax=axes[1], label="NormHV (lower is better)")

plt.tight_layout()
plt.show()

## 4. Convergence Analysis

Analyze how objectives and parameters evolve over evaluations.

In [ ]:
# Objective convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, obj in enumerate(["Epsilon", "NormHypervolume"]):
    ax = axes[i]
    grouped = objectives_df.groupby("evaluation")[obj].agg(["mean", "min", "max", "std"])
    grouped = grouped.sort_index()
    
    ax.fill_between(grouped.index, grouped["min"], grouped["max"], alpha=0.2, color='blue')
    ax.plot(grouped.index, grouped["mean"], 'b-', linewidth=2, label="Mean")
    ax.plot(grouped.index, grouped["min"], 'g--', linewidth=1.5, label="Best")
    ax.set_xlabel("Evaluation")
    ax.set_ylabel(obj)
    ax.set_title(f"Convergence of {obj}")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Parameter convergence for top parameters
top_corr_params = corr_df.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, param in enumerate(top_corr_params):
    ax = axes[i]
    grouped = configs_df.groupby("evaluation")[param].agg(["mean", "std", "min", "max"])
    grouped = grouped.sort_index()
    
    ax.fill_between(grouped.index, grouped["min"], grouped["max"], alpha=0.2)
    ax.plot(grouped.index, grouped["mean"], 'b-', linewidth=2)
    ax.set_xlabel("Evaluation")
    ax.set_ylabel(param)
    ax.set_title(f"Evolution of {param}")

plt.tight_layout()
plt.show()

## 5. Parameter Importance Summary

In [ ]:
# Bar plot of parameter importance
importance_df = corr_df.head(15).copy()

fig, ax = plt.subplots(figsize=(12, 8))

x = np.arange(len(importance_df))
width = 0.35

bars1 = ax.barh(x - width/2, importance_df["Epsilon_corr"], width, label="Epsilon", color='steelblue')
bars2 = ax.barh(x + width/2, importance_df["NormHV_corr"], width, label="NormHV", color='darkorange')

ax.set_xlabel("Spearman Correlation")
ax.set_ylabel("Parameter")
ax.set_title("Parameter Correlation with Objectives (Top 15)")
ax.set_yticks(x)
ax.set_yticklabels(importance_df.index)
ax.legend()
ax.axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Final summary
print("="*60)
print("META-OPTIMIZATION ANALYSIS SUMMARY")
print("="*60)
print(f"\nDataset: {RESULTS_DIR}")
print(f"Total configurations analyzed: {len(configs_df)}")
print(f"Evaluation range: {configs_df['evaluation'].min()} - {configs_df['evaluation'].max()}")
print(f"\nBest Epsilon achieved: {objectives_df['Epsilon'].min():.4f}")
print(f"Best NormHypervolume achieved: {objectives_df['NormHypervolume'].min():.4f}")
print(f"\nPCA: {n_90} components explain 90% of variance")
print(f"\nTop 5 most influential parameters:")
for i, (param, row) in enumerate(corr_df.head(5).iterrows(), 1):
    print(f"  {i}. {param}: |corr| = {row['abs_mean_corr']:.3f}")